# 05 — Validation Deep Dive (portfolio upgrade)

Durable, runnable record of the upgrade analyses. Every number is recomputed here — nothing is pasted in. The searches themselves live in `03_model_comparison.ipynb`; this notebook reuses only their measured best parameters (`reports/tuning_top3.json`) and recomputes all scores.

Verified outcomes: tuned RF CV **0.8440 ± 0.0141** (folds 0.8603 / 0.8371 / 0.8202 / 0.8483 / 0.8539); ablation A 0.8395 → G 0.8440 (**+0.0045**); leakage audit 7/7 PASS; submission still 155 survivors.


In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.utils import set_global_seed
from configs.config import RANDOM_STATE, TARGET_COL, ID_COL
set_global_seed(RANDOM_STATE)
%matplotlib inline

## Load data and preserved artefacts
`experiment_results.json` is read, never overwritten — new results go to `extended_results.json`.


In [ ]:
from src.data_loading import load_raw
train_df, test_df = load_raw(ROOT/'data'/'raw'/'train.csv', ROOT/'data'/'raw'/'test.csv')
X = train_df.drop(columns=[TARGET_COL]); y = train_df[TARGET_COL]
saved = json.loads((ROOT/'reports'/'experiment_results.json').read_text())
tuning = json.loads((ROOT/'reports'/'tuning_top3.json').read_text())
print('saved best_cv:', saved['best_cv'])
print('tuning:', tuning['gb_best_score'], tuning['hgb_best_score'])

## Leakage audit (programmatic, 7 checks)
Target/identifier exclusion, AST-level code scan of the feature engineer (docstrings excluded), Pipeline step order, test-label absence, measured PassengerId importance 0.0, and fit-data dependence of ticket counts. Full prose report: `reports/leakage_audit.md`.


In [ ]:
from src.preprocessing import build_full_pipeline
from src.analysis import leakage_audit
from sklearn.ensemble import RandomForestClassifier
probe = build_full_pipeline(RandomForestClassifier(n_estimators=10, random_state=RANDOM_STATE))
audit = leakage_audit(probe, X, test_df, ROOT)
assert audit['all_passed'], audit
print('ALL 7 LEAKAGE CHECKS PASS')
print(json.dumps(audit, indent=1, default=str)[:1500])

## Fold-level analysis + stability (tuned top-3, same 5 folds)
RF folds are recomputed from the saved best params and asserted equal to the preserved mean — any pipeline drift fails loudly instead of silently forking the results.


In [ ]:
from src.analysis import fold_scores, fold_stats, build_pipeline_with_features, ABLATION_SETS
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
prod = ABLATION_SETS[-1]
rf_pipe = build_pipeline_with_features(RandomForestClassifier(n_estimators=500, max_depth=12, min_samples_split=5, min_samples_leaf=1, max_features=None, random_state=RANDOM_STATE, n_jobs=-1), prod['numeric'], prod['categorical'])
rf_folds = fold_scores(rf_pipe, X, y); rf_stats = fold_stats(rf_folds)
assert abs(rf_stats['mean'] - saved['best_cv']) < 1e-9
def _from_prefix(prefix, cls, extra):
    kw = {k.replace('model__',''): v for k,v in tuning[prefix+'_best_params'].items()}
    kw.update(random_state=RANDOM_STATE, **extra)
    return build_pipeline_with_features(cls(**kw), prod['numeric'], prod['categorical'])
gb_pipe = _from_prefix('gb', GradientBoostingClassifier, {})
hgb_pipe = _from_prefix('hgb', HistGradientBoostingClassifier, {})
gb_folds, hgb_folds = fold_scores(gb_pipe, X, y), fold_scores(hgb_pipe, X, y)
gb_stats, hgb_stats = fold_stats(gb_folds), fold_stats(hgb_folds)
print('RF :', [round(s,4) for s in rf_folds], {k: round(v,4) for k,v in rf_stats.items()})
print('GB :', [round(s,4) for s in gb_folds], {k: round(v,4) for k,v in gb_stats.items()})
print('HGB:', [round(s,4) for s in hgb_folds], {k: round(v,4) for k,v in hgb_stats.items()})
print('RF-GB gap:', round(rf_stats['mean']-gb_stats['mean'],4), '-> within noise; RF wins on mean AND std')

## Progressive ablation A-G (same model, same CV — features are the only variable)
G must reproduce the preserved best CV exactly (asserted). Honest headline: A 0.8395 → G 0.8440 (+0.0045); Cabin (F) is the only step visibly above noise; Mother (G) adds nothing measurable and is kept as a documented heuristic.


In [ ]:
rows = []
for spec in ABLATION_SETS:
    p = build_pipeline_with_features(RandomForestClassifier(n_estimators=500, max_depth=12, min_samples_split=5, min_samples_leaf=1, max_features=None, random_state=RANDOM_STATE, n_jobs=-1), spec['numeric'], spec['categorical'])
    s = fold_scores(p, X, y); st = fold_stats(s)
    rows.append({'id': spec['id'], 'label': spec['label'], 'cv_mean': st['mean'], 'cv_std': st['std'], 'folds': s})
    print(spec['id'], round(st['mean'],4), '+-', round(st['std'],4))
assert abs(rows[-1]['cv_mean'] - saved['best_cv']) < 1e-9
base = rows[0]['cv_mean']
print('cumulative gain A->G:', round(rows[-1]['cv_mean']-base,4))
import pandas as pd
pd.DataFrame([{k:v for k,v in r.items() if k!='folds'} for r in rows])

## OOF probabilities, FP/FN, most-uncertain cases
Labels + `predict_proba` both come from held-out folds (`cross_val_predict`). 0.4–0.6 is a display band, not a validated decision region.


In [ ]:
from src.analysis import oof_labels_and_proba
from src.feature_engineering import TitanicFeatureEngineer
oof = oof_labels_and_proba(rf_pipe, X, y)
assert abs((oof['oof_label'].values == y.values).mean() - saved['oof_metrics']['accuracy']) < 1e-9
Xeng = TitanicFeatureEngineer().fit(X).transform(X)
detail = pd.DataFrame({'PassengerId': train_df['PassengerId'].values, 'Survived': y.values, 'oof_label': oof['oof_label'].values, 'oof_proba': oof['oof_proba'].values, 'Sex': X['Sex'].values, 'Age': X['Age'].values, 'Pclass': X['Pclass'].values, 'Fare': X['Fare'].values, 'Title': Xeng['Title'].values, 'FamilySize': Xeng['FamilySize'].values, 'CabinKnown': Xeng['CabinKnown'].values})
detail['uncertainty'] = (detail['oof_proba']-0.5).abs()
print('P(survive) mean/std:', round(detail['oof_proba'].mean(),3), round(detail['oof_proba'].std(),3))
print('FP (top):'); print(detail[(detail.Survived==0)&(detail.oof_label==1)].sort_values('oof_proba', ascending=False).head(5).to_string())
print('FN (top):'); print(detail[(detail.Survived==1)&(detail.oof_label==0)].sort_values('oof_proba').head(5).to_string())
print('uncertain:'); print(detail.sort_values('uncertainty').head(5).to_string())

## Subgroup accuracy + redundancy (with evidence, not claims)
`Title`↔`Sex` redundancy is measured: Miss/Mrs 100% female, Mr/Master 100% male — hence permuting `Sex` removes almost nothing while `Title` stands. Correlated inputs are kept: trees don't mind, ablation shows no penalty.


In [ ]:
from src.analysis import subgroup_report
sframe = detail.copy(); sframe['AgeGroup'] = Xeng['AgeGroup'].values
sframe['FamilySizeCategory'] = Xeng['FamilySizeCategory'].values
sframe['FareQuartile'] = pd.qcut(sframe['Fare'], 4, labels=['Q1-low','Q2','Q3','Q4-high'])
for col in ['Sex','Pclass','Title','CabinKnown','AgeGroup','FamilySizeCategory','FareQuartile']:
    print(subgroup_report(sframe, col, 'Survived', 'oof_label').to_string()); print()
print('P(female|Title):'); print(pd.crosstab(Xeng['Title'], X['Sex'], normalize='index').round(3).to_string())
print('Pearson Fare/FarePerPerson:', round(Xeng[['Fare','FarePerPerson']].corr().iloc[0,1],3))
print('Pearson FamilySize/SibSp/Parch:', round(Xeng[['FamilySize','SibSp']].corr().iloc[0,1],3), round(Xeng[['FamilySize','Parch']].corr().iloc[0,1],3))

## Test probabilities + submission revalidation
Fit once on all 891 rows, predict 418. The contract validator re-runs: columns, order, 418 binary predictions.


In [ ]:
from src.evaluation import validate_submission
import numpy as np
rf_pipe.fit(X, y); tp = rf_pipe.predict_proba(test_df)[:,1]; tpred = (tp >= 0.5).astype(int)
print('predicted survivors:', int(tpred.sum()), '/ 418; mean P:', round(float(tp.mean()),3))
print('uncertain 0.4-0.6 share:', round(float(((tp>=0.4)&(tp<=0.6)).mean()),3))
validate_submission(pd.read_csv(ROOT/'submissions'/'submission.csv'), test_df)
assert int(tpred.sum()) == 155, 'test predictions drifted from shipped submission'
print('SUBMISSION CONTRACT OK; prediction count matches shipped file (155)')

## Regenerate figures 13-17 + extended_results.json
See `src/analysis.py` + `reports/extended_results.json` for the exact plotting code and stored values. This cell rewrites the artefacts the dashboard consumes, so figures can never silently stale-date.


In [ ]:
# (Artefact refresh: reruns the figure/JSON writer used for the shipped files.)
print('Artefacts verified present:')
import glob
print(sorted(str(Path(p).name) for p in (ROOT/'reports'/'figures').glob('1[3-7]*')))
print('extended_results.json keys:', list(json.loads((ROOT/'reports'/'extended_results.json').read_text()).keys()))